# Pixel Ecosystem Classification with AlphaEarth Foundations Embeddings and a Linear Probe

[Google DeepMind AlphaEarth Foundations](https://deepmind.google/blog/alphaearth-foundations-helps-map-our-planet-in-unprecedented-detail/) is a geospatial embedding model from Google DeepMind. It produces a 64-dimensional "embedding" for every 10m pixel on Earth's land and coastal surface, for each year 2017-2025. Each embedding summarises information from many satellite and geophysical data sources (optical, radar, LiDAR, climate simulations, and more) into a single compact vector that captures the character of that location for that year.

From Google DeepMind:
> "AlphaEarth Foundations provides a powerful new lens for understanding our planet by solving two major challenges: data overload and inconsistent information."

Because the embeddings encode a huge amount of information, they are very effective input features for simple machine learning classifiers - often more effective than raw satellite bands, and with far less data to load and process.

This notebook demonstrates how to:

* Load AlphaEarth Foundations embeddings as a cloud native Zarr dataset, hosted on [Source Cooperative](https://source.coop/tge-labs/aef-mosaic)
* Combine the embeddings with a point-based training dataset of ecosystem classes
* Train a linear probe (logistic regression) classifier and use it to map ecosystems across an area of interest

The training data is a set of labelled points over coastal and marine ecosystems in Nusa Tenggara Barat (West Nusa Tenggara), Indonesia, produced for the [Piksel-Ina notebooks](https://github.com/piksel-ina/piksel-notebooks). It labels points as Mangrove, Photic Coral Reef, Seagrass, Deep Water, Forest, Urban, or Sand.

This notebook follows a similar workflow to the [Sentinel-2 machine learning example](https://github.com/auspatious/coastal-applications-workshop/blob/main/notebooks/examples/05_Sentinel-2_MachineLearning.ipynb) from the Coastal Applications Workshop, but swaps out Sentinel-2 bands (and indices) for AlphaEarth embeddings as the input features, and uses a different model. This provides a richer, but less explicit dataset.

Some great embedding resources:
- Visualisation: https://tilebucket.xyz/designs/latent-color
- Visualisation: https://developmentseed.org/deck.gl-raster/examples/aef-mosaic/
- Introduction: https://developmentseed.org/blog/2026-10-02-earth-embeddings/
- Overview of how embeddings can be made: https://narrative.allen.ai/olmoearth-embeddings


## Set up

Import the required Python libraries:

* `xarray` lazily opens the AlphaEarth embeddings Zarr store, so only the data we need is downloaded
* `odc-geo` (`odc.geo`) handles coordinate reference systems, cropping, and interactive maps (`.odc.explore()`)
* `numpy` is used for maths on the arrays
* `geopandas` loads and maps the training points
* `scikit-learn` provides `PCA` (to view the 64 embedding bands as an RGB image) and `LogisticRegression` (our classifier)
* `matplotlib` converts colour names to RGBA values for the classified map
* `folium` and `ipyleaflet` provide the interactive map layer control and satellite basemap

In [ ]:
import folium
import geopandas as gpd
import matplotlib.colors as mcolors
import numpy as np
import odc.geo.xr
import xarray as xr
from ipyleaflet import basemaps
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

## 1. Load training data

The training data is a GeoJSON file of labelled points, hosted in the Piksel-Ina notebooks repository. Each point has a `Class` (a numeric code) and an `Ecosys_Typ` (a human readable ecosystem type).

We load it using `geopandas`.

In [ ]:
training_points = gpd.read_file("./NusaTenggaraBarat_tdata.geojson")
print(f"Loaded {len(training_points)} training points")

# A colour for each ecosystem type, used in all maps below
ECOSYSTEM_COLORS = {
    "Deep Water": "blue",
    "Forest": "lightgreen",
    "Mangrove": "green",
    "Photic Coral Reef": "pink",
    "Sand": "yellow",
    "Seagrass": "turquoise",
    "Urban": "grey",
}

training_points.explore(
    column="Ecosys_Typ",
    categories=list(ECOSYSTEM_COLORS),
    cmap=list(ECOSYSTEM_COLORS.values()),
    legend=True,
    tiles=basemaps.Esri.WorldImagery,
    style_kwds={"radius": 3},
)

## 2. Define the area of interest

We use the bounding box of the training points as our area of interest. The training points and the AlphaEarth embeddings are in CRS WGS84 so use latitude, longitude, and degrees as units.

In [ ]:
crs = training_points.crs
print(crs)

# total_bounds is (min x, min y, max x, max y) of all the points
aoi_box = odc.geo.geom.box(*training_points.total_bounds, crs=crs)
aoi_box.explore(tiles=basemaps.Esri.WorldImagery)

## 3. Load AlphaEarth Foundations embeddings

The AlphaEarth Foundations mosaic is published on Source Cooperative by Taylor Geospatial as a cloud native [Zarr](https://zarr.dev/) store, accessed here through its `zarr.json` metadata document. Zarr is a chunked, compressed, cloud native array format which plays a similar role for multi-dimensional arrays that COGs (with STAC metadata) play for individual images.

Because it's cloud native, we can open the dataset lazily with `xarray` and only stream the chunks we need for our area of interest, without downloading the full global archive (which would be extremely slow and impractical).

See more about Zarr and cloud native geospatial formats here: https://guide.cloudnativegeo.org/zarr/intro.html

In [ ]:
# Takes ~1 minute. Only metadata is read here, no pixel data is downloaded yet.
embeddings = xr.open_zarr(
    "s3://us-west-2.opendata.source.coop/tge-labs/aef-mosaic/",
    storage_options={"anon": True},  # Public data, no authentication needed
    consolidated=False,
    chunks="auto",  # Let dask pick chunk sizes, important for performance with this very large dataset
    zarr_format=3,
)

### 3.1 Explore the dataset structure

AlphaEarth Foundations embeddings have 64 bands per 10m pixel, with one layer per year.

Note the dimensions: 9 times (years 2017-2025), 64 bands (the embedding), and y and x (the pixel coordinates).

In [ ]:
embeddings = embeddings.odc.assign_crs(crs)  # Both datasets are in WGS84, this just makes it explicit
embeddings

### 3.2 Subset to the area of interest and most recent year

We use `odc.geo.xr.crop` to cut the global dataset down to our area of interest, and select the year 2025.

Compare the dimensions below with those above: `time` is gone (we selected one year), and `y` and `x` went from millions of pixels to thousands. The 64 bands are still there.

This is fast because no pixel data has been loaded yet. We are still building a lazy (dask) task graph.

In [ ]:
embeddings_clipped = odc.geo.xr.crop(embeddings, aoi_box).sel(time=2025)["embeddings"]
embeddings_clipped

### 3.3 Load the data

`.compute()` downloads the pixels, so the data is no longer lazy. Only our subset is loaded, not the global dataset. This takes ~3 minutes, depending on bandwidth and AOI size.

In [ ]:
embeddings_clipped = embeddings_clipped.compute()

### 3.4 Visualise three of the embedding bands

A 64-band embedding can't be viewed directly as an image, but we can view any three bands as red, green and blue. They show similarities and differences across the area, but we can't map them to ecosystem types yet.

To explore any bands anywhere, see https://developmentseed.org/deck.gl-raster/examples/aef-mosaic/.

In [ ]:
bands = ["A22", "A27", "A31"]
embeddings_clipped.sel(band=bands).to_dataset(dim="band").odc.explore(
    tiles=basemaps.Esri.WorldImagery, bands=bands
)

### 3.5 De-quantize the embeddings

To save space, the embeddings are stored as 8-bit integers (`int8`, -127 to 127), with -128 meaning "no data". The original values are floats between -1 and 1. They were quantized by taking the square root (keeping the sign) and multiplying by 127.5, which spreads -1 to 1 across the int8 range (255 / 2 = 127.5). See the [dataset documentation](https://source.coop/tge-labs/aef).

We reverse this to get the float values back: `float = (int / 127.5)² × sign(int)`. PCA and the classifier both need these float values.

In [ ]:
NODATA = -128  # int8 value used for pixels with no data
SCALE = 127.5  # Quantization scale factor (255 / 2), from the dataset documentation

embeddings_float = embeddings_clipped.astype("float32")
embeddings_float = (embeddings_float / SCALE) ** 2 * np.sign(embeddings_float)
embeddings_float = embeddings_float.where(embeddings_clipped != NODATA)  # Set no data pixels to NaN

### 3.6 Make a table of pixels

scikit-learn expects a 2D table with one row per sample and one column per feature. `stack` combines the `y` and `x` dimensions into a single `pixel` dimension, giving one row per pixel and one column per embedding band. The values are unchanged, only their arrangement.

scikit-learn can't handle NaN, so we fill no data pixels with 0 and mask them out of the results afterwards.

In [ ]:
pixels = embeddings_float.stack(pixel=("y", "x")).transpose("pixel", "band")
pixels_filled = pixels.fillna(0)
has_data = embeddings_float.notnull().all("band")  # True for pixels with data, shape (y, x)
pixels

### 3.7 Principal Component Analysis (PCA) false-colour composite

PCA finds the three combinations of the 64 bands that capture the most variation, which we show as red, green and blue. Areas with similar colours have similar embeddings, and so similar land or seascape characteristics.

In [ ]:
pca = PCA(n_components=3)
pca.fit(pixels.dropna("pixel")[::20])  # Fit on every 20th pixel with data: much faster, near-identical result
pca_values = pca.transform(pixels_filled)

# Turn the table back into an image. The component order (b, r, g) is an arbitrary choice of colours.
pca_da = xr.DataArray(
    pca_values,
    coords={"pixel": pixels.pixel, "component": ["b", "r", "g"]},
    dims=["pixel", "component"],
).unstack("pixel")
pca_da = pca_da.where(has_data)

# Scale each component to 0-1 so it can be shown as a colour
pca_da = (pca_da - pca_da.min(["x", "y"])) / (pca_da.max(["x", "y"]) - pca_da.min(["x", "y"]))
pca_rgb = pca_da.odc.assign_crs(crs).to_dataset(dim="component")

Note that the colors in this PCA visualisation do not yet map to any ecosystem type, they just show us similarities and differences in this area. Note the clear blue, purple, yellow, and green areas. Scroll up to our training data to imagine how these colours could represent the 7 classes in the training data.

In [ ]:
pca_rgb.odc.explore(bands=["r", "g", "b"], tiles=basemaps.Esri.WorldImagery)

## 4. Prepare the training data

We look up the embedding pixel nearest to each training point, and put the values in a table with the point's `Class` code. Each row is a training point: the first column is the class code and the other 64 are the embedding bands.

In [ ]:
# Find the embedding pixel nearest to each training point
training_values = embeddings_float.sel(
    x=xr.DataArray(training_points.geometry.x.values, dims="point"),
    y=xr.DataArray(training_points.geometry.y.values, dims="point"),
    method="nearest",
)

training_table = training_values.transpose("point", "band").to_pandas()
training_table.insert(0, "Class", training_points["Class"].values)
training_table = training_table.dropna()  # Drop points that fall on no data pixels
training_table.head()

## 5. Train a linear probe classifier

Foundation models like AlphaEarth are trained so that their embedding space is (close to) *linear*: pixels with similar characteristics sit close together, and different classes can be separated by straight lines (hyperplanes) in the 64-dimensional space. So a simple linear classifier on top of the embeddings, known as a "linear probe", is enough. It is the standard way to evaluate and use foundation model embeddings.

We use logistic regression, which learns one weight per embedding band for each class. It is fast, has almost no settings to tune, and gives a probability for each class. A non-linear model like a Random Forest also works well, but a linear probe shows how much of the work the embeddings already do.

Learn more about logistic regression in the [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

In [ ]:
features = training_table.drop(columns="Class").values
labels = training_table["Class"].values

model = LogisticRegression(
    class_weight="balanced",  # Weight classes equally, even though some have more training points
    max_iter=1000,  # Allow more iterations than the default (100) so training converges
)
model.fit(features, labels)

## 6. Predict across the area of interest

We predict a class for every row (pixel) of the pixel table, then turn the result back into an image.

In [ ]:
predicted = model.predict(pixels_filled)

predicted_da = xr.DataArray(predicted, coords={"pixel": pixels.pixel}, dims="pixel").unstack("pixel")
predicted_da = predicted_da.where(has_data, 0)  # 0 = no data (not used as a class code)
predicted_da = predicted_da.odc.assign_crs(crs)
predicted_da

## 7. Visualise the results

Finally, we bring together the PCA false-colour composite, the predicted classes, and the original training points on a single interactive map.

In [ ]:
# Colour lookup table: row N is the RGBA colour (0-255) for class code N.
class_names = dict(zip(training_points["Class"], training_points["Ecosys_Typ"]))
color_lut = np.zeros((max(class_names) + 1, 4), dtype="uint8")
for code, name in class_names.items():
    color_lut[code] = np.array(mcolors.to_rgba(ECOSYSTEM_COLORS[name])) * 255

m = pca_rgb.odc.explore(
    bands=["r", "g", "b"], tiles=basemaps.Esri.WorldImagery, name="3. PCA (RGB)", layer_control=False
)
predicted_da.odc.explore(map=m, cmap=color_lut, name="2. Predicted Ecosystem Type")
training_points.explore(
    m=m,
    column="Ecosys_Typ",
    categories=list(ECOSYSTEM_COLORS),
    cmap=list(ECOSYSTEM_COLORS.values()),
    name="1. Training Data",
    legend=True,
    legend_kwds={"caption": "Ecosystem Type"},
    marker_kwds={"radius": 4},
    style_kwds={"color": "black", "weight": 1, "fillOpacity": 0.9},
)
folium.LayerControl(collapsed=False, sortLayers=True).add_to(m)
m

## Conclusion and next steps

This notebook showed how AlphaEarth Foundations embeddings, streamed directly from a cloud native Zarr store on Source Cooperative, can be combined with a set of labelled training data points to classify 10m pixel-level ecosystems without ever needing to load or process a raw satellite image.

As a next step, you may like to:

1. Try a different area of interest and/or use your own training points
2. Compare classification outputs against the Sentinel-2-based approach used in the [coastal applications workshop](https://github.com/auspatious/coastal-applications-workshop)
3. Use a ground-truthed dataset to validate the output
4. Do a change analysis. Run this for 2017 and for 2025 and then analyse how the area of interest has changed. You could use [this](https://knowledge.dea.ga.gov.au/notebooks/How_to_guides/Land_cover_change_mapping/) as a base.